## Part 2, Random Forests and SVMs

In [32]:
# importing libraries

import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import f1_score, classification_report


In [38]:
# Importing Dataframe and sense checking

h10k = pd.read_csv('hansard10000.csv')

print(h10k.shape)
print(h10k.isnull().sum())
h10k.head(3)

(10000, 8)
speech             0
party            411
constituency     411
date               0
speech_class       0
major_heading      0
year               0
speakername        0
dtype: int64


,speech,party,constituency,date,speech_class,major_heading,year,speakername
0,We will now suspend for three minutes for sani...,Conservative,Ribble Valley,2021-03-11,Speech,Contingencies Fund (No. 2) Bill,2021,Nigel Evans
1,I am now beginning to share the indignation of...,Labour,City of Chester,2020-11-24,Speech,Exiting the European Union,2020,Christian Matheson
2,"The hon. Lady is wrong; as a matter of fact, w...",Conservative,Newark,2021-02-10,Speech,Building Safety,2021,Robert Jenrick


In [20]:
# Listing out the different parties, the different entries in 'speech_class'
# counting the nr. of speeches less thatn 1000 characters

print(h10k['party'].value_counts(), '\n')
print(h10k['speech_class'].value_counts(), '\n')
print("Nr. of speeches under 1000 ch.:", (h10k['speech'].str.len() < 1000).sum())

party
Conservative                        6192
Labour                              1828
Scottish National Party              560
Labour (Co-op)                       280
Liberal Democrat                     221
Speaker                              208
Democratic Unionist Party            140
Independent                           58
Plaid Cymru                           51
Social Democratic & Labour Party      21
Green Party                           17
Alliance                              12
Alba Party                             1
Name: count, dtype: int64 

speech_class
Speech        9614
Procedural     343
Division        43
Name: count, dtype: int64 

Nr. of speeches under 1000 ch.: 7752


In [40]:
# Replacing 'Labour Co-op' with 'Labour', dropping all parties except top 4, sense checking

h10k['party'] = hrd10k['party'].replace('Labour (Co-op)', 'Labour')
h10k = h10k[h10k['party'].isin(['Conservative', 'Labour', 'Scottish National Party', 'Liberal Democrat'])]

print(h10k.isnull().sum())
h10k['party'].value_counts()

speech           0
party            0
constituency     0
date             0
speech_class     0
major_heading    0
year             0
speakername      0
dtype: int64


party
Conservative               6192
Labour                     2108
Scottish National Party     560
Liberal Democrat            221
Name: count, dtype: int64

In [30]:
# Removing ll non 'speech' entries from 'speech class'
# Removing all speeches under 1000 characters and sense checking

h10k = h10k[h10k['speech_class'].isin(['Speech'])]

h10k = h10k[h10k['speech'].str.len() > 1000]

print('Shape of the dataframe:', h10k.shape)

h10k.head()


Shape of the dataframe: (2111, 8)


,speech,party,constituency,date,speech_class,major_heading,year,speakername
2,"The hon. Lady is wrong; as a matter of fact, w...",Conservative,Newark,2021-02-10,Speech,Building Safety,2021,Robert Jenrick
5,"Given that this was outlined earlier today, it...",Conservative,Great Yarmouth,2021-04-13,Speech,Northern Ireland,2021,Brandon Lewis
7,The aftermath of the 2008 crisis saw not only ...,Labour,"Sheffield, Hallam",2021-02-23,Speech,Government's Management of the Economy,2021,Olivia Blake
17,The right hon. Gentleman puts it correctly. Wh...,Conservative,Sutton and Cheam,2020-12-07,Speech,United Kingdom Internal Market Bill,2020,Paul Scully
18,"As always, I am grateful to the hon. Gentleman...",Conservative,North East Somerset,2021-01-28,Speech,Business of the House,2021,Jacob Rees-Mogg


In [33]:
# Vectorizing the data and setting up Random Forest and SVM

vec_h10k = TfidfVectorizer(stop_words = 'english', max_features = 3000)            # Setting parameters for vectors
x = vec_h10k.fit_transform(h10k['speech'])
y = h10k['party']

x_train, x_test, y_train, y_test = train_test_split(x, y, stratify = y, random_state = 26)       # Splitting the set

rando = RandomForestClassifier(n_estimators = 300, random_state = 26).fit(x_train, y_train)      # Random Forest
support_vm = SVC(kernel = 'linear', random_state = 26).fit(x_train, y_train)                     # Support Vector Machine

In [41]:
# Classification of parties and F1 score

for name, clf in [('RandomForest', rando), ('SVM', support_vm)]:
    party_pred = clf.predict(x_test)
    print(name)
    print('Macro avg. F1:', f1_score(y_test, party_pred, average = 'macro', zero_division = 0))
    print(classification_report(y_test, party_pred, zero_division = 0))

RandomForest
Macro avg. F1: 0.415070121650697
                         precision    recall  f1-score   support

           Conservative       0.70      0.96      0.81       312
                 Labour       0.71      0.39      0.51       157
       Liberal Democrat       0.00      0.00      0.00        18
Scottish National Party       0.82      0.22      0.35        41

               accuracy                           0.70       528
              macro avg       0.56      0.39      0.42       528
           weighted avg       0.69      0.70      0.65       528

SVM
Macro avg. F1: 0.44991938899021994
                         precision    recall  f1-score   support

           Conservative       0.76      0.89      0.82       312
                 Labour       0.61      0.55      0.58       157
       Liberal Democrat       0.00      0.00      0.00        18
Scottish National Party       0.67      0.29      0.41        41

               accuracy                           0.71       528


In [42]:
# Vectorizing the data and setting up Random Forest and SVM

vec_h10k_ngram = TfidfVectorizer(stop_words = 'english', max_features = 3000, ngram_range = (1, 3))     # Setting parameters for vectors
x1 = vec_h10k_ngram.fit_transform(h10k['speech'])
y1 = h10k['party']

x1_train, x1_test, y1_train, y1_test = train_test_split(x1, y1, stratify = y1, random_state = 26)       # Splitting the set

rando1 = RandomForestClassifier(n_estimators = 300, random_state = 26).fit(x1_train, y1_train)      # Random Forest
support_vm1 = SVC(kernel = 'linear', random_state = 26).fit(x1_train, y1_train)                     # Support Vector Machine

In [43]:
# Classification of parties and F1 score

for name1, clf in [('RandomForest', rando1), ('SVM', support_vm1)]:
    party_pred1 = clf.predict(x1_test)
    print(name1)
    print('Macro avg. F1:', f1_score(y1_test, party_pred1, average = 'macro', zero_division = 0))
    print(classification_report(y1_test, party_pred1, zero_division = 0))

RandomForest
Macro avg. F1: 0.41746949793668114
                         precision    recall  f1-score   support

           Conservative       0.79      0.94      0.86      1549
                 Labour       0.63      0.48      0.54       527
       Liberal Democrat       0.00      0.00      0.00        55
Scottish National Party       0.70      0.16      0.27       140

               accuracy                           0.76      2271
              macro avg       0.53      0.40      0.42      2271
           weighted avg       0.73      0.76      0.73      2271

SVM
Macro avg. F1: 0.4770302937788669
                         precision    recall  f1-score   support

           Conservative       0.81      0.92      0.86      1549
                 Labour       0.65      0.56      0.60       527
       Liberal Democrat       0.00      0.00      0.00        55
Scottish National Party       0.68      0.33      0.44       140

               accuracy                           0.78      2271

### Creating a custom tokenizer

In [44]:
# Importing necessary libraries

import re
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.corpus import stopwords

lemmatizer = WordNetLemmatizer()                  # Setting some variables for the function
stop_w = set(stopwords.words('english'))

In [49]:
def custom_toke(text):
    ''' 
    Takes words and tokenizes them 
    First puts all letters into lowercase, then drops anything that's not letters
    Lemmatizes for consistency across similar words
    Removes stopwords
    '''

    text = text.lower()
    toke = re.findall(r'[a-z]+', text)
    toke = [t for t in toke if len(t) > 2]
    toke = [lemmatizer.lemmatize(t) for t in toke]
    toke = [t for t in toke if t not in stop_w]
    
    return toke

In [54]:
# Testing out the new tokenizer

vec_h10k_custom = TfidfVectorizer(tokenizer = custom_toke, max_features = 2000)       # Setting parameters for vectors

x2 = vec_h10k_custom.fit_transform(h10k['speech'])
y2 = h10k['party']

x2_train, x2_test, y2_train, y2_test = train_test_split(x2, y2, stratify = y2, random_state = 26)       # Splitting the set

rando2 = RandomForestClassifier(n_estimators = 300, random_state = 26).fit(x2_train, y2_train)      # Random Forest
support_vm2 = SVC(kernel = 'linear', random_state = 26).fit(x2_train, y2_train)                     # Support Vector Machine

C:\Users\nikki\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [55]:
# Classification of parties and F1 score

for name2, clf in [('RandomForest', rando2), ('SVM', support_vm2)]:
    party_pred2 = clf.predict(x2_test)
    print(name2)
    print('Macro avg. F1:', f1_score(y2_test, party_pred2, average = 'macro', zero_division = 0))
    print(classification_report(y2_test, party_pred2, zero_division = 0))

RandomForest
Macro avg. F1: 0.4075049229762141
                         precision    recall  f1-score   support

           Conservative       0.78      0.97      0.86      1549
                 Labour       0.68      0.42      0.52       527
       Liberal Democrat       0.00      0.00      0.00        55
Scottish National Party       0.70      0.15      0.25       140

               accuracy                           0.77      2271
              macro avg       0.54      0.38      0.41      2271
           weighted avg       0.73      0.77      0.72      2271

SVM
Macro avg. F1: 0.46472256551636715
                         precision    recall  f1-score   support

           Conservative       0.81      0.94      0.87      1549
                 Labour       0.67      0.53      0.59       527
       Liberal Democrat       0.00      0.00      0.00        55
Scottish National Party       0.71      0.28      0.40       140

               accuracy                           0.78      2271